# 09c: Cardio-70k, development and freezing (gates G1 and G3)

Design decisions
- Task C: Cardio-70k (70,000 examinations, 11 predictors, outcome recorded at examination, about half with the outcome). Label-independent splits with seed 42 and the shares of notebook 06 (training portion 40%, split into dev-train 75% and dev-cal 25%; calibration split 1 30%; calibration split 2 15%; test 15%). Only training-portion rows are kept in memory; the explained subsamples are chosen from row indices alone.
- Targets, fixed before any Cardio model was fitted and scaled to the 50% base rate (on the odds scale the enrichment target is as strict as BRFSS Heart and the negative-outcome target is stricter than both BRFSS tasks): negative-outcome claims at alpha_0 = 0.20 (grid 0.10, 0.20, 0.30); enrichment claims at beta = 0.80, so alpha_1 = 0.20 (grid 0.6, 0.7, 0.8). The plan's grid (alpha_0 0.02, 0.05, 0.10; beta 0.2, 0.3, 0.5) is evaluated too and reported as descriptive rows. delta_1 = delta_2 = 0.025, M = 10.
- Everything else follows notebook 05: gate G1 against a tuned unconstrained XGBoost; gate G3 runs the whole SCAX pipeline inside development with the same cascade and the same utility rule (at least half of the development patients receive a certified negative-outcome explanation at the main target); alpha_sel per kind is the smallest value on its grid (negative outcome 0.20, 0.25, 0.30; enrichment 0.20, 0.25, 0.30, 0.35) that the development selective error passes at the expected number of accepted patients; explained subsamples of 5,000 test patients and of calibration split 2 sized from the development acceptance rates.
- Weight has decimal values, so every XGBoost export converts inputs to float32 exactly as XGBoost does; the final model is checked against the native library on every training row and by the primary and second soundness checks on 300 boxes (Z3 on 30).
- The frozen configuration is written to configs/frozen/cardio70k.json and its SHA-256 to results/09c_frozen.json, and this notebook commits before notebook 09d opens calibration split 1. Development results are stored on Drive, so a rerun resumes; a rerun must reproduce the same frozen configuration.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import subprocess, sys, warnings


def quiet_warnings():
    # imodels switches on every DeprecationWarning; Colab's message code then warns on every output line
    warnings.filterwarnings('ignore', category=DeprecationWarning, module='jupyter_client')
    warnings.filterwarnings('ignore', message=r'datetime\.datetime\.utcnow')


quiet_warnings()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'z3-solver'], check=True)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE_ROOT = '/content/drive/MyDrive/SCAX_Research'
REPO_DIR = f'{DRIVE_ROOT}/scax-research'
for name in ('.gitconfig', '.git-credentials'):
    if os.path.exists(f'{DRIVE_ROOT}/{name}'):
        shutil.copy(f'{DRIVE_ROOT}/{name}', os.path.expanduser(f'~/{name}'))
os.chdir(REPO_DIR)
os.environ['SCAX_DRIVE_ROOT'] = DRIVE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')
pull = subprocess.run(['git', 'pull', 'origin', 'main'], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())

In [ ]:
import json, time, math
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from scax import paths
from scax.utils import seed_everything, sha256_array, sha256_file, write_json
from scax.data import public, splits
from scax.models import train
from scax.models.predictor import native_scores
from scax.experiments import development as dev
from scax.experiments import model_ablations as ma
from scax.explain.candidates import CandidateGenerator
from scax.explain.verify import verify_boxes, drop_unverified, z3_sample
from scax.encode.interval_bound import IntervalBound
from scax.encode.cell_search import CellSearch
from scax.encode.real_search import RealSearch
from scax.encode.real_encoding import Z3RealEncoding
from scax.certify.clopper_pearson import cp_upper
from scax.frozen import write_frozen, load_frozen
import inspect
if 'input_cast' not in inspect.signature(dev.fit_exported).parameters:
    raise RuntimeError('The stage 13 source is not loaded. Run the stage 13 batch, then Runtime > Disconnect and delete runtime, then run again.')

SEED = 42
seed_everything(SEED)
TASK = 'cardio70k'
M, DELTA1, DELTA2 = 10, 0.025, 0.025
ALPHA0_MAIN, ALPHA0_GRID = 0.20, [0.10, 0.20, 0.30]
BETA_MAIN, BETA_GRID = 0.80, [0.6, 0.7, 0.8]
PLAN_ALPHA0_GRID, PLAN_BETA_GRID = [0.02, 0.05, 0.10], [0.2, 0.3, 0.5]
ALL_ALPHA0, ALL_BETA = sorted(set(ALPHA0_GRID + PLAN_ALPHA0_GRID)), sorted(set(BETA_GRID + PLAN_BETA_GRID))
LAMBDAS, M_LIST = [0.5, 0.25], [1, 5, 10]
TARGET_NEGATIVE_RELEASE, G1_MAX_LOSS = 0.50, 0.01
N_PRIOR_DEV, N_EVAL_DEV, N_PRIOR_FINAL, N_TEST_EXPLAIN, N2_CAP = 1000, 1000, 2000, 5000, 12000
ALPHA_SEL_GRID = {0: [0.20, 0.25, 0.30], 1: [0.20, 0.25, 0.30, 0.35]}
INPUT_CAST = 'float32'
DEV_DIR = paths.DERIVED / 'development'; DEV_DIR.mkdir(parents=True, exist_ok=True)
FROZEN_DIR = paths.DERIVED / 'frozen'; FROZEN_DIR.mkdir(parents=True, exist_ok=True)
CARDS = json.load(open(f'{paths.RESULTS}/01_dataset_cards.json'))
assert json.load(open(f'{paths.RESULTS}/06_encoding_tests.json'))['gate_G2']['passed']

## Splits, training portion only, and the development partition

In [ ]:
X, y = public.LOADERS[TASK](paths.RAW)
assert sha256_file(public.raw_dir(paths.RAW, TASK) / public.SPECS[TASK].file) == CARDS['sha256'][TASK], 'Cardio file differs from notebook 01'
code = splits.make_splits(len(y), SEED)
split_file = paths.DERIVED / 'splits' / f'{TASK}.npy'
split_file.parent.mkdir(parents=True, exist_ok=True)
if split_file.exists():
    assert np.array_equal(np.load(split_file), code), 'stored Cardio split differs'
else:
    np.save(split_file, code)
SPLIT = splits.summary(code)
train_rows = splits.indices(code, 'train')
Xt, yt, names = X.to_numpy(dtype=float)[train_rows], y[train_rows], list(X.columns)
del X, y                                    # rows outside the training portion are not kept
pos = lambda idx: np.searchsorted(train_rows, idx)
dtr, dcal = pos(splits.indices(code, 'dev_train')), pos(splits.indices(code, 'dev_cal'))
rng = np.random.default_rng(SEED)
perm = rng.permutation(dcal)
n_cert = int(round(0.8 * len(dcal)))
cert, pool = np.sort(perm[:n_cert]), np.sort(perm[n_cert:])
ev = np.sort(rng.choice(pool, N_EVAL_DEV, replace=False))
pr = np.sort(rng.choice(dtr, N_PRIOR_DEV, replace=False))
P = {'dev_train': dtr, 'dev_cal': dcal, 'cert': cert, 'eval': ev, 'prior': pr}
print('splits', SPLIT)
print({k: len(v) for k, v in P.items()}, 'training rows', len(train_rows), 'with the outcome', int(yt.sum()))

## Gate G1: tuned unconstrained reference

In [ ]:
t0 = time.time()
REF = dev.tuned_reference(Xt[P['dev_train']], yt[P['dev_train']], Xt[P['dev_cal']], yt[P['dev_cal']], SEED)
G1_REF = float(REF.auroc.max())
REF.insert(0, 'task', TASK)
REF.to_csv(f'{paths.RESULTS}/09c_g1_reference.csv', index=False)
print(f'best tuned AUROC {G1_REF:.4f} ({time.time() - t0:.0f}s)')
print(REF.head(3).round(4).to_string(index=False))

## Gate G3: the SCAX pipeline inside development, in the plan order

In [ ]:
DEV_TABLES, DEV_META, evaluated, CHOICE = [], [], [], None


def main_rows(tab):
    return tab[(tab.M == M) & np.isclose(tab.alpha_0, ALPHA0_MAIN) & np.isclose(tab.beta, BETA_MAIN)]


for name, params in dev.CASCADE:
    f_tab, f_meta = DEV_DIR / f'{TASK}__{name}.csv', DEV_DIR / f'{TASK}__{name}.json'
    if f_tab.exists() and f_meta.exists():
        tab, meta = pd.read_csv(f_tab), json.load(open(f_meta))
        print(f'{name}: loaded from Drive')
    else:
        quick = train.fit('xgboost', Xt[P['dev_train']], yt[P['dev_train']], params, SEED)
        auroc = roc_auc_score(yt[P['dev_cal']], native_scores(quick, 'xgboost', Xt[P['dev_cal']]))
        g1 = bool(auroc >= G1_REF - G1_MAX_LOSS)
        if not g1:
            tab, meta = pd.DataFrame(), {'config': name, 'params': params, 'auroc_dev_cal': auroc}
        else:
            tab, meta = dev.evaluate_config(name, params, Xt, yt, P['dev_train'], P['cert'], P['eval'], P['prior'],
                                            names, SEED, M, DELTA1, DELTA2 / 2, ALL_ALPHA0, ALL_BETA,
                                            [1.0] + LAMBDAS, M_LIST, input_cast=INPUT_CAST)
        meta.update({'task': TASK, 'g1_reference': G1_REF, 'g1_pass': g1})
        tab.to_csv(f_tab, index=False)
        json.dump(meta, open(f_meta, 'w'), indent=1, default=float)
    DEV_META.append(meta)
    if not meta['g1_pass']:
        print(f"{name}: fails G1 (AUROC {meta['auroc_dev_cal']:.4f} against {G1_REF:.4f})"); continue
    if 'task' not in tab.columns:
        tab.insert(0, 'task', TASK)
    DEV_TABLES.append(tab)
    mr = main_rows(tab)
    for lam in LAMBDAS:
        rate = float(mr[mr['lambda'] == lam].release_rate_class0.iloc[0])
        evaluated.append((rate, name, params, lam))
        print(f"{name} lambda={lam}: negative-outcome release rate {rate:.3f} at alpha_0 = {ALPHA0_MAIN}")
        if rate >= TARGET_NEGATIVE_RELEASE and CHOICE is None:
            CHOICE = {'config': name, 'params': params, 'lambda': lam, 'target_met': True}
    if CHOICE is not None:
        break
if CHOICE is None:
    rate, name, params, lam = max(evaluated, key=lambda t: t[0])
    CHOICE = {'config': name, 'params': params, 'lambda': lam, 'target_met': False}
print('frozen choice:', CHOICE)
DEV_ALL = pd.concat(DEV_TABLES, ignore_index=True)
DEV_ALL.to_csv(f'{paths.RESULTS}/09c_dev_results.csv', index=False)
pd.DataFrame(DEV_META).to_csv(f'{paths.RESULTS}/09c_dev_configs.csv', index=False)

In [ ]:
show = DEV_ALL[(DEV_ALL.M == M) & np.isclose(DEV_ALL.beta, BETA_MAIN)]
print(show[['config', 'lambda', 'alpha_0', 'release_rate', 'release_rate_class0', 'release_rate_class1',
            'selective_error_class0', 'selective_upper_class0', 'selective_error_class1', 'selective_upper_class1',
            'released_size_class0', 'released_support_class0']].round(3).to_string(index=False))
print(pd.DataFrame(DEV_META)[['config', 'g1_pass', 'auroc_dev_cal', 'g1_reference', 'thresholds',
                              'median_seconds_per_patient', 'distinct_boxes']].round(4).to_string(index=False))

## Policy targets and the size of the explained subsample of calibration split 2

In [ ]:
r = DEV_ALL[(DEV_ALL.config == CHOICE['config']) & (DEV_ALL['lambda'] == CHOICE['lambda']) & (DEV_ALL.M == M)
            & np.isclose(DEV_ALL.alpha_0, ALPHA0_MAIN) & np.isclose(DEV_ALL.beta, BETA_MAIN)].iloc[0]
n_cal2 = int((code == splits.SPLIT_CODES['cal2']).sum())
need = [math.ceil(2000 / r.release_rate) if r.release_rate > 0 else N2_CAP]
for k in (0, 1):
    rr = r[f'release_rate_class{k}']
    need.append(math.ceil(1000 / rr) if rr > 0 else N2_CAP)
N2 = int(min(N2_CAP, n_cal2, max(need)))
ALPHA_SEL, PROJECTION = {}, {}
for k in (0, 1):
    a_exp = int(round(r[f'release_rate_class{k}'] * N2))
    rate = r[f'selective_error_class{k}'] if a_exp > 0 and not np.isnan(r[f'selective_error_class{k}']) else 1.0
    u = float(cp_upper(int(round(rate * a_exp)), a_exp, DELTA2 / 2)) if a_exp > 0 else 1.0
    ALPHA_SEL[k] = dev.smallest_at_or_above(u, ALPHA_SEL_GRID[k])
    PROJECTION[k] = {'expected_accepted': a_exp, 'development_selective_error': float(rate), 'projected_upper': u}
print('explained patients of calibration split 2:', N2, '| alpha_sel', ALPHA_SEL)
print(json.dumps(PROJECTION, indent=1, default=float))

## Final model on the whole training portion

In [ ]:
t0 = time.time()
model, ens, pred = dev.fit_exported(Xt, yt, CHOICE['params'], SEED, names, INPUT_CAST)
ens.meta.update({'task': TASK, 'config': CHOICE['config'], 'params': CHOICE['params'], 'seed': SEED,
                 'tau_rule': 'Youden on 5-fold out-of-fold scores of the training portion'})
model_file = FROZEN_DIR / f'{TASK}__model.json'
ens.save(model_file)
AGREE = ma.exported_vs_native_detail(model, 'xgboost', pred, Xt)
g = pred.grid
ib = IntervalBound(pred); cs = CellSearch(ib); rs = RealSearch(ens, g.thresholds)
zr = Z3RealEncoding(ens, g.thresholds, timeout_ms=20000)
rng = np.random.default_rng(SEED)
pick = rng.choice(len(Xt), 300, replace=False)
C = g.cells(Xt[pick]); yh = pred.predict_cells(C)
CHECKS = {'boxes': 300, 'primary_vs_second_disagree': 0, 'z3_checked': 0, 'z3_unknown': 0, 'z3_disagree': 0, 'sound': 0}
for i in range(len(C)):
    box = list(g.point_box(C[i]))
    for j in rng.choice(g.active, size=int(rng.integers(3, min(18, len(g.active)) + 1)), replace=False):
        box[j] = (0, int(g.K[j]))
    box = tuple(box); yb = int(yh[i])
    a = cs.check(box, yb); b = rs.check(box, yb)
    CHECKS['sound'] += a == 'sound'
    CHECKS['primary_vs_second_disagree'] += ('unknown' not in (a, b)) and a != b
    if i < 30:
        c = zr.check(box, yb); CHECKS['z3_checked'] += 1; CHECKS['z3_unknown'] += c == 'unknown'
        CHECKS['z3_disagree'] += (c != 'unknown') and (a != 'unknown') and c != a
assert AGREE['beyond_rounding'] == 0 and CHECKS['primary_vs_second_disagree'] == 0 and CHECKS['z3_disagree'] == 0, 'final model fails G2'
FINAL = {'model_sha256': ens.hash(), 'exported_vs_native': AGREE, 'soundness_checks': CHECKS, 'thresholds': int(g.K.sum()),
         'log10_family_size': g.family_size_log10(), 'seconds': time.time() - t0}
print(json.dumps(FINAL, indent=1, default=float))

## Frozen prior from 2,000 training patients

In [ ]:
gen = CandidateGenerator(pred, M=M, seed=SEED)
cache_path = FROZEN_DIR / f'{TASK}__prior_cache.pkl'
gen.load(cache_path)
rng = np.random.default_rng(SEED + 2)
prr = np.sort(rng.choice(len(Xt), N_PRIOR_FINAL, replace=False))
C = g.cells(Xt[prr]); yh = pred.predict_cells(C)
t0 = time.time()
lists = dev.candidate_sets(gen, C, yh, log=print, every=500)
gen.save(cache_path)
boxes = sorted({(c.key, c.y) for v in lists for c in v})
verdicts, _ = verify_boxes(g, boxes)
sub = {gen.cell_key(C[i]): gen.cache[gen.cell_key(C[i])] for i in range(len(C))}
removed = drop_unverified(sub, verdicts)
lists = [sub[gen.cell_key(C[i])] for i in range(len(C))]
z3res, _ = z3_sample(g, [b for b in boxes if verdicts[b] == 'sound'], 30, SEED, timeout_ms=10000, time_budget_s=120)
counts = dev.prior_counts(lists, M)
prior = {'type': 'mixture', 'lambda': CHOICE['lambda'], 'd': int(g.d), 'K': g.K.tolist(), 'M': M,
         'training_patients': N_PRIOR_FINAL, 'training_rows': [int(train_rows[i]) for i in prr],
         'counts': [[[list(t) for t in k], v] for k, v in sorted(counts.items())]}
prior_file = FROZEN_DIR / f'{TASK}__prior.json'
json.dump(prior, open(prior_file, 'w'))
FINAL.update({'prior_sha256': sha256_file(prior_file), 'prior_boxes': len(counts), 'prior_unverified_removed': int(removed),
              'prior_z3_sample': z3res, 'prior_seconds': time.time() - t0,
              'prior_median_seconds_per_patient': float(np.median([gen.seconds[gen.cell_key(c)] for c in C]))})
print(f"prior: {len(counts)} boxes, removed {removed}, Z3 sample {z3res}, {time.time() - t0:.0f}s")

## Explained subsamples (row indices only) and the frozen configuration

In [ ]:
rng = np.random.default_rng(SEED + 1)
test_explained = np.sort(rng.choice(splits.indices(code, 'test'), N_TEST_EXPLAIN, replace=False))
cal2_explained = np.sort(rng.choice(splits.indices(code, 'cal2'), N2, replace=False))
np.save(FROZEN_DIR / f'{TASK}__test_explained.npy', test_explained)
np.save(FROZEN_DIR / f'{TASK}__cal2_explained.npy', cal2_explained)
cfg = {
    'task': TASK, 'outcome': public.SPECS[TASK].outcome, 'outcome_definition': public.SPECS[TASK].outcome_definition,
    'data_sha256': CARDS['sha256'][TASK], 'split_sha256': SPLIT['sha256'], 'seed': SEED,
    'model': {'family': 'xgboost', 'config': CHOICE['config'], 'params': CHOICE['params'], 'sha256': FINAL['model_sha256'],
              'tau': ens.tau, 'tau_rule': 'Youden on 5-fold out-of-fold scores of the training portion',
              'input_cast': INPUT_CAST, 'file': f'datasets/derived/frozen/{TASK}__model.json',
              'trained_on': 'training portion (dev-train and dev-cal)'},
    'candidates': {'M': M, 'order_seed': SEED, 'node_budget': 200000,
                   'inflation': 'binary search per side, every sound visited box kept',
                   'soundness': 'interval pre-check, then exact leaf branch-and-bound; every candidate re-checked by the second exact implementation'},
    'prior': {'type': 'mixture', 'lambda': CHOICE['lambda'], 'training_patients': N_PRIOR_FINAL,
              'file': f'datasets/derived/frozen/{TASK}__prior.json', 'sha256': FINAL['prior_sha256']},
    'targets': {'delta_1': DELTA1, 'delta_2': DELTA2, 'alpha_0': ALPHA0_MAIN, 'alpha_0_grid': ALPHA0_GRID,
                'beta': BETA_MAIN, 'beta_grid': BETA_GRID, 'alpha_1': round(1 - BETA_MAIN, 10),
                'plan_alpha_0_grid': PLAN_ALPHA0_GRID, 'plan_beta_grid': PLAN_BETA_GRID,
                'scaling': 'targets scaled to the 50% base rate, chosen before any Cardio model was fitted',
                'alpha_sel': {'negative_outcome': ALPHA_SEL[0], 'enrichment': ALPHA_SEL[1]},
                'policy': 'negative-outcome and enrichment releases certified separately at delta_2 / 2 each'},
    'release_rule': 'fewest constrained features, then most calibration support, then tightest bound; abstain if no candidate passes',
    'explained': {'test': N_TEST_EXPLAIN, 'test_sha256': sha256_array(test_explained),
                  'cal2': int(len(cal2_explained)), 'cal2_sha256': sha256_array(cal2_explained)},
    'development': {'g1_reference_auroc': G1_REF, 'target_negative_release': TARGET_NEGATIVE_RELEASE,
                    'target_met': CHOICE['target_met'], 'release_rate_negative': float(r.release_rate_class0),
                    'release_rate_enrichment': float(r.release_rate_class1),
                    'selective_error_negative': float(r.selective_error_class0),
                    'selective_error_enrichment': float(r.selective_error_class1), 'alpha_sel_projection': PROJECTION},
    'frozen_model_checks': {k: FINAL[k] for k in ('exported_vs_native', 'soundness_checks', 'thresholds', 'log10_family_size',
                                                  'prior_boxes', 'prior_unverified_removed', 'prior_z3_sample')},
    'calibration_split_1_opened': False,
}
cfg_file, rec = Path(f'{REPO_DIR}/configs/frozen/{TASK}.json'), Path(f'{paths.RESULTS}/09c_frozen.json')
if rec.exists() and cfg_file.exists():          # rerun after freezing: the stored configuration is kept and must match
    old, FROZEN_SHA = load_frozen(cfg_file, json.load(open(rec))['frozen_config_sha256'][TASK])
    same = lambda v: json.loads(json.dumps(v, default=float))
    differ = [k for k in ('data_sha256', 'split_sha256', 'model', 'candidates', 'prior', 'targets', 'explained') if same(cfg[k]) != old[k]]
    assert not differ, f'a rerun must reproduce the frozen configuration; differs in {differ}'
    print('rerun: the stored frozen configuration is kept (model, prior, targets and subsamples reproduced)')
else:
    FROZEN_SHA = write_frozen(cfg_file, cfg)
write_json(rec, {'frozen_config_sha256': {TASK: FROZEN_SHA}, 'written_before_calibration_split_1': True, 'choice': CHOICE,
                 'n_cal2_explained': N2, 'alpha_sel': ALPHA_SEL, 'splits': SPLIT})
print(FROZEN_SHA)
print(json.dumps({k: cfg[k] for k in ('model', 'prior', 'targets', 'explained', 'development')}, indent=1, default=float))

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
  "09c: Cardio-70k splits, development (gates G1 and G3), policy targets, final model, prior, explained subsamples and frozen configuration, before calibration split 1"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)